In [1]:
from collections import Counter
from dataclasses import dataclass
from enum import Enum
from typing import Protocol

import numpy as np
import plotly.express as px
from IPython.display import clear_output


# utils

In [2]:
def plot_map(map: np.ndarray, replace: bool = False, highlight: tuple[int, int] | None = None, highlight_label: str | None = None) -> None:
    """Display a probability heatmap.

    Parameters
    ----------
    map : np.ndarray
        2D probability array.
    replace : bool
        If True, clear previous output before showing the figure.
    highlight : tuple[int,int] | None
        Optional (row, col) coordinate to mark on the heatmap.
    highlight_label : str | None
        Optional text label shown next to the highlight marker.
    """
    if replace:
        clear_output(wait=True)

    fig = px.imshow(
        map,
        color_continuous_scale="Viridis",
        text_auto=".1%",
        aspect="auto",
        origin="upper"
    )

    if highlight is not None:
        row, col = highlight
        if not (0 <= row < map.shape[0] and 0 <= col < map.shape[1]):
            raise ValueError("highlight coordinate out of bounds")

        # Use an open circle marker so the cell text remains readable
        fig.add_scatter(
            x=[col],
            y=[row],
            mode="markers+text" if highlight_label else "markers",
            marker={
                "color": "rgba(0,0,0,0)",
                "size": 48,
                "symbol": "circle",
                "line": {"width": 3, "color": "red"}
            },
            text=[highlight_label] if highlight_label else None,
            textposition="top center",
            showlegend=False,
        )

    fig.update_layout(
        title="Probability Heatmap",
        xaxis_title="X",
        yaxis_title="Y",
        coloraxis_colorbar_title="Probability",
    )

    fig.show()


# Game

In [3]:
map = np.arange(1, 26, dtype=float).reshape(5, 5)
map = map / map.sum()
plot_map(map, highlight=(2, 3), highlight_label="Target")

In [4]:
class PingResult(Enum):
    DETECTED = 1
    NOT_DETECTED = 0

class PingSensor:
    """
    This represent a ping sensor.
    It can ping a location and return whether the ping was detected or not.

    If the sub is at the location, the ping will be detected with a probability of 0.8.
    If the sub is 1 unit away, the ping will be detected with a probability of 0.25.
    If the sub is 2 or more units away, the ping will be detected with a probability of 0.05.

    Based on that specification, the ping method is created.

    """

    def __init__(self, sub_location: tuple[int, int], map_arr: np.ndarray):
        self.map_arr = map_arr
        self.rows, self.cols = map_arr.shape
        self.sub_location = sub_location


    def ping(self, pinged_location: tuple[int, int]) -> PingResult:
        x, y = pinged_location
        index_in_range = 0 <= x < self.rows and 0 <= y < self.cols
        if not index_in_range:
            raise ValueError("Coordinates out of bounds")

        dist = self._calculate_distance(pinged_location, self.sub_location)
        p_detect = self._calc_p_detect_given_distance(dist)

        return PingResult.DETECTED if np.random.rand() < p_detect else PingResult.NOT_DETECTED


    def calc_p_of_outcome_for_LocationPinged_given_Location(
            self,
            outcome: PingResult,
            location_j: tuple[int, int],
            location_pinged: tuple[int, int],
            ):
        """
        this method represent 'reading' the sensors spec.
        Examples:
        P(ND_a|B) - probability of getting NonDetect for A, given sub is in B
        P(D_a|B) - probability of getting Detect for A, given sub is in B
        """

        dist = self._calculate_distance(location_pinged, location_j)
        p_detect_given_distance = self._calc_p_detect_given_distance(dist)

        match outcome:
            case PingResult.DETECTED:
                return p_detect_given_distance
            case PingResult.NOT_DETECTED:
                return 1 - p_detect_given_distance
            case _:
                raise ValueError(f"Unknown outcome: {outcome}")


    def _calc_p_detect_given_distance(self, dist: float):
        if dist < 1:
            return 0.8
        elif dist <= np.sqrt(2):
            return 0.25
        else:
            return 0.001


    def _calculate_distance(self, pinged_location: tuple[int, int], sub_location: tuple[int, int]):
        p_x, p_y = pinged_location
        s_x, s_y = sub_location

        return np.sqrt((p_x - s_x) ** 2 + (p_y - s_y) ** 2)



In [5]:
# Sanity checks for _calculate_distance
sensor = PingSensor(sub_location=(2, 2), map_arr=map)

assert sensor._calculate_distance((2, 2), (2, 2)) == 0
assert sensor._calculate_distance((2, 4), (2, 2)) == 2
assert sensor._calculate_distance((0, 0), (2, 2)) == np.sqrt(8)

assert np.isclose(sensor._calculate_distance((1, 1), (2, 2)), np.sqrt(2))
assert np.isclose(sensor._calculate_distance((2, 1), (2, 2)), 1)
assert np.isclose(sensor._calculate_distance((4, 0), (2, 2)), np.sqrt(8))

print("distance sanity checks passed")

distance sanity checks passed


In [6]:
print("pinging (2, 2) 100 times...")
print(Counter([sensor.ping((2, 2)) for i in range(100)]))
print("pinging (1, 1) 100 times...")
print(Counter([sensor.ping((1, 1)) for i in range(100)]))
print("pinging (0, 0) 100 times...")
print(Counter([sensor.ping((0, 0)) for i in range(100)]))

pinging (2, 2) 100 times...
Counter({<PingResult.DETECTED: 1>: 90, <PingResult.NOT_DETECTED: 0>: 10})
pinging (1, 1) 100 times...
Counter({<PingResult.NOT_DETECTED: 0>: 73, <PingResult.DETECTED: 1>: 27})
pinging (0, 0) 100 times...
Counter({<PingResult.NOT_DETECTED: 0>: 100})


In [7]:
class PingStrategy(Protocol):
    """Protocol — any class with this interface works."""

    def choose_location(self, map_arr: np.ndarray) -> tuple[tuple[int, int], float, str]:
        ...

class MostLikelySquareStrategy(PingStrategy):
    def __init__(self):
        self.name = "MostLikelySquareStrategy"

    def choose_location(self, map_arr: np.ndarray) -> tuple[tuple[int, int], float, str]:
        best_location_to_ping = np.unravel_index(np.argmax(map_arr), map_arr.shape)
        best_prob = map_arr[best_location_to_ping]

        return (best_location_to_ping, best_prob, self.name)

class RandomSquareStrategy(PingStrategy):
    def __init__(self):
        self.name = "RandomSquareStrategy"

    def choose_location(self, map_arr: np.ndarray) -> tuple[tuple[int, int], float, str]:
        rows, cols = map_arr.shape
        random_location = (np.random.randint(rows), np.random.randint(cols))
        random_prob = map_arr[random_location]

        return (random_location, random_prob, self.name)


In [8]:
class InformationGainStrategy(PingStrategy):

    def __init__(self, sensor: PingSensor):
        self.name = "InformationGainStrategy"
        self.sensor = sensor


    def choose_location(
        self,
        map_arr: np.ndarray
    ) -> tuple[tuple[int, int], float, str]:

        # H(P)
        current_entropy = self.calc_entropy(map_arr)

        information_gains = np.zeros(map_arr.shape)

        # Consider every possible square we could ping
        for row_i in range(map_arr.shape[0]):
            for col_i in range(map_arr.shape[1]):

                ping_location = (row_i, col_i)


                # ------------------------------------------------
                # What would the belief map look like if we got D?
                # ------------------------------------------------

                map_if_detected, p_detected = self.get_new_belief_map(
                    map_arr,
                    ping_location,
                    PingResult.DETECTED
                )

                entropy_if_detected = self.calc_entropy(
                    map_if_detected
                )


                # ------------------------------------------------
                # What would the belief map look like if we got ND?
                # ------------------------------------------------

                map_if_not_detected, p_not_detected = self.get_new_belief_map(
                    map_arr,
                    ping_location,
                    PingResult.NOT_DETECTED
                )

                entropy_if_not_detected = self.calc_entropy(
                    map_if_not_detected
                )


                # ------------------------------------------------
                # Expected entropy after pinging this location
                #
                # E[H_after]
                #
                # = P(D) * H(P | D)
                # + P(ND) * H(P | ND)
                # ------------------------------------------------

                expected_entropy_after_ping = (
                    p_detected
                    * entropy_if_detected
                    +
                    p_not_detected
                    * entropy_if_not_detected
                )


                # ------------------------------------------------
                # Information Gain
                #
                # IG = H(before) - E[H(after)]
                # ------------------------------------------------

                information_gains[row_i, col_i] = (
                    current_entropy
                    - expected_entropy_after_ping
                )


        best_location_to_ping = np.unravel_index(
            np.argmax(information_gains),
            information_gains.shape
        )

        best_prob = map_arr[best_location_to_ping]

        return (
            best_location_to_ping,
            best_prob,
            self.name
        )


    def calc_entropy(self, map_arr: np.ndarray) -> float:
        probabilities = map_arr[map_arr > 0]

        return -np.sum( probabilities * np.log2(probabilities) )


    def calc_p_outcome(
        self,
        map_arr: np.ndarray,
        pinged_location: tuple[int, int],
        outcome: PingResult
    ) -> float:

        """
        Calculates:

        P(outcome | pinged_location)

        by summing over every possible submarine location:

        P(outcome | submarine_location, pinged_location)
        *
        P(submarine_location)
        """

        p_outcome = 0.0

        for row_i in range(map_arr.shape[0]):
            for col_i in range(map_arr.shape[1]):

                submarine_location = (row_i, col_i)

                prior = map_arr[submarine_location]

                likelihood = (
                    self.sensor
                    .calc_p_of_outcome_for_LocationPinged_given_Location(
                        outcome,
                        submarine_location,
                        pinged_location
                    )
                )

                p_outcome += likelihood * prior

        return p_outcome


    def get_new_belief_map(
        self,
        map_arr: np.ndarray,
        pinged_location: tuple[int, int],
        outcome: PingResult
    ) -> np.ndarray:

        # ------------------------------------------------
        # P(outcome)
        # ------------------------------------------------

        p_outcome = self.calc_p_outcome(
            map_arr,
            pinged_location,
            outcome
        )


        # ------------------------------------------------
        # Bayes update
        #
        # P(L | outcome)
        #
        # = P(outcome | L) * P(L)
        #   -------------------------
        #          P(outcome)
        # ------------------------------------------------

        new_beliefs = np.zeros_like(
            map_arr,
            dtype=float
        )

        for row_i in range(map_arr.shape[0]):
            for col_i in range(map_arr.shape[1]):

                submarine_location = (row_i, col_i)

                prior = map_arr[submarine_location]

                likelihood = (
                    self.sensor
                    .calc_p_of_outcome_for_LocationPinged_given_Location(
                        outcome,
                        submarine_location,
                        pinged_location
                    )
                )

                new_beliefs[submarine_location] = (
                    likelihood
                    * prior
                    / p_outcome
                )

        return new_beliefs, p_outcome

In [9]:
# # initial
# map = np.arange(1, 26, dtype=float).reshape(5, 5)
# map = map / map.sum()

# # random
# rng = np.random.default_rng()
# rng_map = rng.random((5, 5))
# rng_map = rng_map / rng_map.sum()

# # best
# best_map = np.zeros((5, 5), dtype=float)
# best_map[2, 2] = 1.0
# best_map[3, 3] = 1.0
# best_map = best_map / best_map.sum()

# plot_map(InformationGainStrategy(sensor = PingSensor(sub_location=(2, 2), map_arr=map)).choose_location(map))
# plot_map(InformationGainStrategy(sensor = PingSensor(sub_location=(2, 2), map_arr=rng_map)).choose_location(rng_map))
# plot_map(InformationGainStrategy(sensor = PingSensor(sub_location=(2, 2), map_arr=best_map)).choose_location(best_map))


In [10]:
@dataclass(frozen=True)
class GameTurnResult:
    strategy_name: str
    game_won: bool
    pinged_location: tuple[int, int]
    pinged_location_prob: float
    ping_outcome: PingResult


class BayesGameState:
    def __init__(
            self,
            sub_location: tuple[int, int],
            map_array: np.ndarray,
            sensor: PingSensor,
            strategy: PingStrategy):
        self.sub_location = sub_location
        self.map_array = map_array

        self.sensor = sensor
        self.strategy = strategy


    def advance_state(self) -> GameTurnResult:
        (location_to_ping, location_to_ping_prob, strategy_name) = self.strategy.choose_location(self.map_array)
        outcome = self.sensor.ping(location_to_ping)

        new_map = self.get_new_belief_map(location_to_ping, outcome)
        self.map_array = new_map

        return GameTurnResult(
            strategy_name = strategy_name,
            game_won=(location_to_ping == self.sub_location and outcome == PingResult.DETECTED),
            pinged_location=location_to_ping,
            pinged_location_prob = location_to_ping_prob,
            ping_outcome=outcome
        )


    def get_new_belief_map(self, pinged_location: tuple[int, int], outcome: PingResult):
        p_outcome = 0
        for row_i, _ in enumerate(self.map_array):
            for col_i, _ in enumerate(self.map_array):
                prior = self.map_array[row_i, col_i]
                p_outcome += (self.sensor.calc_p_of_outcome_for_LocationPinged_given_Location(
                    outcome, pinged_location, (row_i, col_i)) * prior)

        new_beliefs = np.zeros_like(self.map_array, dtype=float)
        for row_i, _ in enumerate(self.map_array):
            for col_i, _ in enumerate(self.map_array):
                p_outcome_given_location = self.sensor.calc_p_of_outcome_for_LocationPinged_given_Location(
                    outcome, pinged_location, (row_i, col_i))
                prior = self.map_array[row_i, col_i]
                new_beliefs[row_i, col_i] = (p_outcome_given_location * prior) / p_outcome

        return new_beliefs


# Play the game

In [11]:
# initial
# sub_location = (2, 2)
# map = np.arange(1, 26, dtype=float).reshape(5, 5)
# map = map / map.sum()

# uniform
map = np.ones((10, 10), dtype=float)
map = map / map.sum()
# sub_location = (4, 3)
sub_location = (
    np.random.randint(map.shape[0]),   # 0..4
    np.random.randint(map.shape[1]),   # 0..4
)

# random
# rng = np.random.default_rng()
# sub_location = (2, 2)
# map = rng.random((5, 5))
# map = map / map.sum()


# sub_location = (2, 2)
# map = np.array([
#     [0.15, 0.08, 0.02, 0.03, 0.02],
#     [0.10, 0.12, 0.03, 0.04, 0.03],
#     [0.04, 0.03, 0.02, 0.03, 0.04],
#     [0.10, 0.12, 0.03, 0.04, 0.03],
#     [0.15, 0.08, 0.02, 0.03, 0.02],
# ], dtype=float)
# map = map / map.sum()

# sub_location = (2, 2)
# map = np.array([
#     [0.02, 0.03, 0.04, 0.08, 0.15],
#     [0.01, 0.02, 0.03, 0.06, 0.12],
#     [0.01, 0.01, 0.02, 0.04, 0.08],
#     [0.02, 0.02, 0.03, 0.04, 0.06],
#     [0.03, 0.04, 0.05, 0.06, 0.08],
# ], dtype=float)
# map = map / map.sum()

sensor = PingSensor(sub_location, map)
strategy = MostLikelySquareStrategy()
# strategy = InformationGainStrategy(sensor)
game_state = BayesGameState(sub_location, map, sensor, strategy)

plot_map(map)

ping_coordinates = []
print(f"Sub position is {sub_location}")

count = 0
while True:
    user_input = input("Press Enter to advance the game state...")
    if user_input.lower() == 'q':
        print("Exiting the game.")
        break

    count += 1
    result = game_state.advance_state()
    print(f"Pinged location: {result.pinged_location}, Outcome: {result.ping_outcome}, Probability: {result.pinged_location_prob:.4f}")
    ping_coordinates.append(result.pinged_location)

    # plot_map(game_state.map_array, replace=False, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")
    plot_map(game_state.map_array, replace=True, highlight=result.pinged_location, highlight_label=f"{result.ping_outcome.name} ({result.pinged_location_prob:.1%})")

    if result.game_won:
        print("Submarine found!")
        print(f"Number of pings: {count}")
        print(Counter(ping_coordinates).most_common())
        break

Exiting the game.


















<!-- # Monte carlo comparisions -->


In [12]:
# ============================================================
# Monte Carlo experiments
# ============================================================

@dataclass(frozen=True)
class GameResult:
    strategy_name: str
    game_won: bool
    num_pings: int


def sample_submarine_location(
    probability_map: np.ndarray
) -> tuple[int, int]:
    """Sample submarine location according to the probability map."""

    probabilities = probability_map.flatten()

    flat_index = np.random.choice(
        len(probabilities),
        p=probabilities
    )

    return np.unravel_index(
        flat_index,
        probability_map.shape
    )


def sample_submarine_location_uniform(
    probability_map: np.ndarray
) -> tuple[int, int]:
    """Sample submarine location uniformly across the map."""

    flat_index = np.random.choice(
        probability_map.size
    )

    return np.unravel_index(
        flat_index,
        probability_map.shape
    )


def run_game(
    strategy: PingStrategy,
    initial_map: np.ndarray,
    sample_location_fn=sample_submarine_location
) -> GameResult:

    sub_location = sample_location_fn(initial_map)

    game_map = initial_map.copy()

    sensor = PingSensor(
        sub_location=sub_location,
        map_arr=game_map
    )

    game_state = BayesGameState(
        sub_location=sub_location,
        map_array=game_map,
        sensor=sensor,
        strategy=strategy
    )

    num_pings = 0

    while True:

        result = game_state.advance_state()

        num_pings += 1

        if result.game_won:
            return GameResult(
                strategy_name=result.strategy_name,
                game_won=True,
                num_pings=num_pings
            )


def run_monte_carlo(
    strategy: PingStrategy,
    initial_map: np.ndarray,
    num_games: int = 10_000,
    sample_location_fn=sample_submarine_location
) -> list[GameResult]:

    return [
        run_game(
            strategy=strategy,
            initial_map=initial_map,
            sample_location_fn=sample_location_fn
        )
        for _ in range(num_games)
    ]


def run_static_search_game(
    initial_map: np.ndarray,
    sample_location_fn=sample_submarine_location
) -> GameResult:

    sub_location = sample_location_fn(initial_map)

    probabilities = initial_map.flatten()

    sorted_indices = np.argsort(-probabilities)

    sub_flat_index = np.ravel_multi_index(
        sub_location,
        initial_map.shape
    )

    rank = np.where(
        sorted_indices == sub_flat_index
    )[0][0]

    return GameResult(
        strategy_name="StaticProbabilityRanking",
        game_won=True,
        num_pings=rank + 1
    )


def run_static_search_monte_carlo(
    initial_map: np.ndarray,
    num_games: int = 10_000,
    sample_location_fn=sample_submarine_location
) -> list[GameResult]:

    return [
        run_static_search_game(
            initial_map=initial_map,
            sample_location_fn=sample_location_fn
        )
        for _ in range(num_games)
    ]


def print_results(
    results: list[GameResult]
) -> None:

    pings = np.array([
        result.num_pings
        for result in results
    ])

    percentiles = np.percentile(
        pings,
        [10, 25, 50, 75, 90, 95, 99]
    )

    print(f"Strategy:       {results[0].strategy_name}")
    print(f"Games:          {len(results):,}")
    print(f"Average pings:  {np.mean(pings):.2f}")
    print(f"Std deviation:  {np.std(pings):.2f}")
    print(f"Minimum pings:  {np.min(pings)}")
    print(f"P10:            {percentiles[0]:.0f}")
    print(f"P25:            {percentiles[1]:.0f}")
    print(f"Median (P50):   {percentiles[2]:.0f}")
    print(f"P75:            {percentiles[3]:.0f}")
    print(f"P90:            {percentiles[4]:.0f}")
    print(f"P95:            {percentiles[5]:.0f}")
    print(f"P99:            {percentiles[6]:.0f}")
    print(f"Maximum pings:  {np.max(pings)}")



In [13]:
# initial
size = 10
map = np.arange(1, size * size + 1, dtype=float).reshape(size, size)
map = map / map.sum()

# random
rng = np.random.default_rng()
map = rng.random((size, size))
map = map / map.sum()

plot_map(map, replace=True)

In [14]:
# ============================================================
# Run experiments
# ============================================================

NUM_GAMES = 100


# ============================================================
# Experiment 1:
# True submarine distribution matches our prior
# ============================================================

most_likely_results = run_monte_carlo(
    strategy=MostLikelySquareStrategy(),
    initial_map=map,
    num_games=NUM_GAMES
)

information_gain_results = run_monte_carlo(
    strategy=InformationGainStrategy(
        sensor=PingSensor(
            sub_location=(0, 0),
            map_arr=map
        )
    ),
    initial_map=map,
    num_games=NUM_GAMES
)

static_results = run_static_search_monte_carlo(
    initial_map=map,
    num_games=NUM_GAMES
)


print("=== Prior matches reality ===")
print()

print_results(most_likely_results)
print()

print_results(information_gain_results)
print()

print_results(static_results)


# ============================================================
# Experiment 2:
# True submarine distribution is uniform
# Our prior is therefore wrong
# ============================================================

most_likely_uniform_truth_results = run_monte_carlo(
    strategy=MostLikelySquareStrategy(),
    initial_map=map,
    num_games=NUM_GAMES,
    sample_location_fn=sample_submarine_location_uniform
)

information_gain_uniform_truth_results = run_monte_carlo(
    strategy=InformationGainStrategy(
        sensor=PingSensor(
            sub_location=(0, 0),
            map_arr=map
        )
    ),
    initial_map=map,
    num_games=NUM_GAMES,
    sample_location_fn=sample_submarine_location_uniform
)

static_uniform_truth_results = run_static_search_monte_carlo(
    initial_map=map,
    num_games=NUM_GAMES,
    sample_location_fn=sample_submarine_location_uniform
)


print()
print()
print("=== Uniform reality, non-uniform prior ===")
print()

print_results(most_likely_uniform_truth_results)
print()

print_results(information_gain_uniform_truth_results)
print()

print_results(static_uniform_truth_results)

KeyboardInterrupt: 